# Enarau Objective 2 — Productivity & Degradation Time Series (LandTrendr)

Component C (vegetation condition) and an extension of component D (historical LULC change) of
the Enarau Conservancy remote-sensing consultancy. Builds Landsat (1984-present), HLS
(2015-present), and Sentinel-2 (2017-present) vegetation-index time series for Enarau, Mbokishi,
and both corridor phases, plus a LandTrendr disturbance/recovery segmentation on dry-season
Landsat NBR (1984-2025). Complements Objective 1's categorical Dynamic World change detection
with continuous vegetation-condition evidence.

Requires a `.env` file in the repo root with `EE_PROJECT=<your-gee-cloud-project-id>` and Earth
Engine authenticated on this machine (`earthengine authenticate`).

## 1. Setup

In [ ]:
import ee
import eetools
import geemap

from eetools.compositing import build_composite
from eetools.constants import (
    HLS_L30_COLLECTION,
    HLS_S30_COLLECTION,
    HLS_SCALE,
    L5_SR_COLLECTION,
    L7_SR_COLLECTION,
    L8_SR_COLLECTION,
    L9_SR_COLLECTION,
    LANDSAT_SCALE,
    S2_SCALE,
    S2_SR_COLLECTION,
)
from eetools.io import export_image_list_to_drive, export_table_to_drive
from eetools.landtrendr import get_change_map, run_landtrendr_from_aoi
from eetools.sensors.chirps.preprocessing import export_total_rainfall_table
from eetools.sensors.hls.preprocessing import get_hls_merged_collection
from eetools.sensors.landsat.preprocessing import (
    get_l5_sr_collection,
    get_l7_sr_collection,
    get_l8_sr_collection,
    get_l9_sr_collection,
)
from eetools.sensors.sentinel.preprocessing import get_s2_sr_collection
from eetools.vectors import get_sites_geometry, vector_files_to_feature_collection
from eetools.visualization.vis_params import SITES_VIS_PARAMS
from eetools.zonal import image_collection_to_region_stats_fc

In [ ]:
try:
    import config
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "config module not found -- run `uv sync` from the repo root to install this "
        "project (including config.py) into the environment."
    ) from exc

In [ ]:
eetools.initialize()

In [ ]:
# Local aliases for readability -- values are single-sourced in config.py.
CRS = config.PROJECT_CRS
EXPORT_FOLDER = config.EXPORT_FOLDER
SEASON_MONTHS = config.SEASON_MONTHS
PERIODS = config.PERIODS
MIN_OBS = config.MIN_OBS
INDEX_BANDS_COMMON = config.INDEX_BANDS_COMMON
S2_INDEX_BANDS_EXTRA = config.S2_INDEX_BANDS_EXTRA
WATER_MASK_HOUSEKEEPING_BANDS = config.WATER_MASK_HOUSEKEEPING_BANDS
VALID_OBS_BAND = config.VALID_OBS_BAND
YEAR_END = config.YEAR_END

## 2. Study-area boundary & project geometry

In [ ]:
site_files = [(site["path"], site["site_id"], site["site_name"]) for site in config.SITES]
sites_fc = vector_files_to_feature_collection(site_files)

# Buffered union of all sites -- one project-wide geometry for every composite/export below,
# not four per-site clips (same convention as Objective 1).
project_geom = get_sites_geometry(sites_fc).buffer(config.STUDY_AREA_BUFFER_M, maxError=10)

## 3. Sensor collection builders

Reuses `eetools.sensors.{landsat,hls,sentinel}.preprocessing` directly for cloud/shadow masking,
band scaling, and index computation -- per the objective-1->2 handoff notes Sec.2, this resolves
the plan's own documented HLS band-name/Fmask gaps (eetools already uses the correct unpadded
HLS band names and applies Fmask masking) and its Sentinel-2 cloud-masking gap (eetools uses
Cloud Probability, not the plan's hand-rolled Cloud Score+ code). `apply_water_masking=True`
(the eetools default) is kept so open-water pixels don't skew the vegetation-condition
composites below -- NDVI and MNDWI are requested from every sensor purely to satisfy that
masking step's own precondition and are not part of the reported index set.

MSAVI2 and CIred_edge are both in `eetools.sensors.indices.INDEX_REGISTRY` (added there since
the objective-1->2 handoff notes flagged them as missing) and requested here like any other
index -- no local index implementation needed. CIred_edge still only resolves for Sentinel-2,
since it needs a red-edge band HLS/Landsat don't have.

Each sensor builder's own date-range validation (`validate_*_date_range`) requires the
requested window to fall *entirely* within that sensor's real coverage over this AOI, not just
overlap it -- `get_available_window` below queries each sensor's actual first/last acquisition
date once (a single `getInfo()` round trip) so the full-record collection can be built once and
reused, rather than guessing satellite operational dates or rebuilding per composite window.

In [ ]:
def get_available_window(collection_ids, aoi: ee.Geometry) -> tuple[ee.Date, ee.Date]:
    """Query a raw EE collection's (or merged collections') actual first/last acquisition dates
    over the AOI.

    One-time getInfo() round trip so each sensor collection builder below can be called with a
    window guaranteed to satisfy eetools' exact-containment date-range validation
    (eetools.utils.validate_collection_date_range), instead of hardcoding satellite operational
    dates that can be off for this specific AOI. `end` is the exact timestamp of the most recent
    scene (matching that same validation), so the single most recent scene is excluded by a
    subsequent `.filterDate(start, end)`'s exclusive upper bound -- immaterial across a
    decades-long record.
    """
    ids = [collection_ids] if isinstance(collection_ids, str) else list(collection_ids)
    col = ee.ImageCollection(ids[0]).filterBounds(aoi)
    for collection_id in ids[1:]:
        col = col.merge(ee.ImageCollection(collection_id).filterBounds(aoi))
    min_ts = col.aggregate_min("system:time_start").getInfo()
    max_ts = col.aggregate_max("system:time_start").getInfo()
    return ee.Date(min_ts), ee.Date(max_ts)

In [ ]:
LANDSAT_SENSOR_BUILDERS = {
    "L5": (get_l5_sr_collection, L5_SR_COLLECTION),
    "L7": (get_l7_sr_collection, L7_SR_COLLECTION),
    "L8": (get_l8_sr_collection, L8_SR_COLLECTION),
    "L9": (get_l9_sr_collection, L9_SR_COLLECTION),
}


def build_landsat_collection() -> ee.ImageCollection:
    """Full-record (1984-present) merged L5/L7/L8/L9 collection, built once. eetools computes
    every requested index (including MSAVI2) directly via calc_indices/INDEX_REGISTRY -- no
    per-sensor band-map handling needed here."""
    requested_indices = INDEX_BANDS_COMMON + WATER_MASK_HOUSEKEEPING_BANDS
    cols = []
    for builder, collection_id in LANDSAT_SENSOR_BUILDERS.values():
        start, end = get_available_window(collection_id, project_geom)
        cols.append(builder(project_geom, start, end, indices=requested_indices))

    merged = cols[0]
    for col in cols[1:]:
        merged = merged.merge(col)
    return merged.filterDate(f"{config.LANDSAT_YEAR_START}-01-01", f"{YEAR_END + 1}-01-01")


landsat_col = build_landsat_collection()

In [ ]:
def build_hls_collection() -> ee.ImageCollection:
    """Full-record (2015-present) merged HLS L30+S30 collection, built once. No red-edge band
    map entry for HLS in eetools, so NDRE/CIred_edge aren't available here (Sentinel-2 provides
    those instead)."""
    requested_indices = INDEX_BANDS_COMMON + WATER_MASK_HOUSEKEEPING_BANDS
    start, end = get_available_window([HLS_L30_COLLECTION, HLS_S30_COLLECTION], project_geom)
    col = get_hls_merged_collection(project_geom, start, end, indices=requested_indices)
    return col.filterDate(f"{config.HLS_YEAR_START}-01-01", f"{YEAR_END + 1}-01-01")


hls_col = build_hls_collection()

In [ ]:
def build_s2_collection() -> ee.ImageCollection:
    """Full-record (2017-present) Sentinel-2 SR Harmonized collection, built once. The only
    sensor with a red-edge band, so NDRE and CIred_edge (both via eetools) are requested here."""
    requested_indices = INDEX_BANDS_COMMON + WATER_MASK_HOUSEKEEPING_BANDS + S2_INDEX_BANDS_EXTRA
    start, end = get_available_window(S2_SR_COLLECTION, project_geom)
    col = get_s2_sr_collection(project_geom, start, end, indices=requested_indices)
    return col.filterDate(f"{config.S2_YEAR_START}-01-01", f"{YEAR_END + 1}-01-01")


s2_col = build_s2_collection()

In [ ]:
SENSOR_SPECS = {
    "landsat": {
        "collection": landsat_col,
        "year_start": config.LANDSAT_YEAR_START,
        "min_obs": MIN_OBS["landsat"],
        "index_bands": INDEX_BANDS_COMMON,
        "scale": LANDSAT_SCALE,
    },
    "hls": {
        "collection": hls_col,
        "year_start": config.HLS_YEAR_START,
        "min_obs": MIN_OBS["hls"],
        "index_bands": INDEX_BANDS_COMMON,
        "scale": HLS_SCALE,
    },
    "sentinel2": {
        "collection": s2_col,
        "year_start": config.S2_YEAR_START,
        "min_obs": MIN_OBS["sentinel2"],
        "index_bands": INDEX_BANDS_COMMON + S2_INDEX_BANDS_EXTRA,
        "scale": S2_SCALE,
    },
}

## 4. Composite & date-window helpers

In [ ]:
def get_date_window(year_start: int, year_end: int, season: str) -> tuple[ee.Date, ee.Date]:
    """Return the (start, end) ee.Date window for a season spanning year_start..year_end.

    A single year is a window with year_start == year_end; a multi-year period (the
    baseline/pre/current comparisons) uses year_start < year_end. Mirrors Objective 1's
    get_date_window -- same wet/dry month bounds -- but reads config.SEASON_MONTHS rather than
    hardcoding them.
    """
    start_month, end_month = SEASON_MONTHS[season]
    if season == "annual":
        return ee.Date.fromYMD(year_start, 1, 1), ee.Date.fromYMD(year_end + 1, 1, 1)
    return ee.Date.fromYMD(year_start, start_month, 1), ee.Date.fromYMD(year_end, end_month + 1, 1)


def build_index_composite(
    collection: ee.ImageCollection, index_bands: list, start: ee.Date, end: ee.Date
) -> ee.Image:
    """Median index composite plus a per-pixel valid-observation count band.

    Mirrors the plan's per-sensor composite builders (Sec.6.4/7.3/8.2), but the count band is
    explicitly cast to Double before being combined with the float index bands -- avoiding the
    `.count()` int/float export bug the objective-1->2 handoff notes (Sec.4) flag as already
    having hit Objective 1's own composite builder (`Pixel type not supported`, then
    `inconsistent types` once cast the wrong way).
    """
    subset = collection.filterDate(start, end)
    composite = build_composite(subset, index_bands, composite_stat="median")
    count = subset.select(index_bands[0]).count().rename(VALID_OBS_BAND).toDouble()
    return composite.addBands(count).clip(project_geom)

## 5. Annual & seasonal composites (per sensor, full record)

Builds one composite per (sensor, year, season) -- wet, dry, and annual -- for each sensor's
full available record (Landsat 1984-2025, HLS 2015-2025, Sentinel-2 2017-2025), plus a
`coverage_flag` band (whether `valid_obs_count` clears the season-appropriate threshold in
`config.MIN_OBS`). Lazy Earth Engine graph construction only -- nothing is computed until a
zonal-stats table or raster export actually runs.

In [ ]:
records = {}
for sensor_name, spec in SENSOR_SPECS.items():
    for year in range(spec["year_start"], YEAR_END + 1):
        for season in SEASON_MONTHS:
            min_obs = spec["min_obs"][season]
            start, end = get_date_window(year, year, season)
            composite = build_index_composite(spec["collection"], spec["index_bands"], start, end)
            composite = composite.addBands(
                composite.select(VALID_OBS_BAND).gte(min_obs).rename("coverage_flag")
            ).set({"sensor_group": sensor_name, "year": year, "season": season})
            records[(sensor_name, year, season)] = composite

print(f"Built {len(records)} sensor/year/season composites.")

## 6. Headline period composites (baseline / pre / current)

Skips a period for a sensor whose record doesn't reach that far back (e.g. the 1984-2000
`baseline` period is Landsat-only; HLS/Sentinel-2 only ever get `pre`/`current` or `current`
respectively).

In [ ]:
period_records = {}
for sensor_name, spec in SENSOR_SPECS.items():
    for period_name, (year_start, year_end) in PERIODS.items():
        if year_start < spec["year_start"]:
            continue
        start, end = get_date_window(year_start, year_end, "annual")
        composite = build_index_composite(spec["collection"], spec["index_bands"], start, end)
        composite = composite.addBands(
            composite.select(VALID_OBS_BAND).gte(spec["min_obs"]["annual"]).rename("coverage_flag")
        ).set({"sensor_group": sensor_name, "period": period_name})
        period_records[(sensor_name, period_name)] = composite

print(f"Built {len(period_records)} sensor/period composites: {sorted(period_records)}")

## 7. Trend maps and pre/post anomaly maps

Per the plan's own guidance (Sec.10.3), slope maps are limited to one productivity metric
(MSAVI2), one moisture/woody-condition metric (NDMI), and one bare-ground metric (BSI) rather
than exported for every index -- Sentinel-2's red-edge NDRE trend is added as a fine-scale
value-add. Reference-normalized (site-minus-Mbokishi) anomalies (plan Sec.10.2) are a per-site
table-level operation on the exported zonal-statistics CSVs, done in pandas in a later step (see
Sec.14 of the plan and Objective 1's own plotting notebook) -- not attempted here in image
space.

In [ ]:
def build_trend_image(sensor_name: str, season: str, index_band: str, year_start: int, year_end: int) -> ee.Image:
    """Per-pixel OLS slope of index_band vs. year across [year_start, year_end] via
    ee.Reducer.linearFit() (plan Sec.10.3, implementation option 1)."""
    imgs = []
    for year in range(year_start, year_end + 1):
        composite = records[(sensor_name, year, season)]
        year_band = ee.Image.constant(year).toFloat().rename("year")
        imgs.append(year_band.addBands(composite.select(index_band).toFloat()))
    fit = ee.ImageCollection(imgs).reduce(ee.Reducer.linearFit())
    return fit.select("scale").rename(f"{index_band}_slope").clip(project_geom)


TREND_SPECS = [
    # (sensor, season, index_band, year_start, year_end, label)
    ("landsat", "dry", "MSAVI2", 1984, 2025, "landsat_dry_1984_2025"),
    ("landsat", "dry", "NDMI", 1984, 2025, "landsat_dry_1984_2025"),
    ("landsat", "dry", "BSI", 1984, 2025, "landsat_dry_1984_2025"),
    ("landsat", "dry", "MSAVI2", 2000, 2025, "landsat_dry_2000_2025"),
    ("landsat", "dry", "NDMI", 2000, 2025, "landsat_dry_2000_2025"),
    ("landsat", "dry", "BSI", 2000, 2025, "landsat_dry_2000_2025"),
    ("hls", "dry", "MSAVI2", 2016, 2025, "hls_dry_2016_2025"),
    ("hls", "dry", "NDMI", 2016, 2025, "hls_dry_2016_2025"),
    ("hls", "dry", "BSI", 2016, 2025, "hls_dry_2016_2025"),
    ("sentinel2", "dry", "NDRE", 2017, 2025, "s2_dry_2017_2025"),
]

trend_maps = {
    (sensor, index_band, label): build_trend_image(sensor, season, index_band, year_start, year_end)
    for sensor, season, index_band, year_start, year_end, label in TREND_SPECS
}
print(f"Built {len(trend_maps)} trend maps.")

In [ ]:
ANOMALY_INDEX_BANDS = ["MSAVI2", "NDMI", "BSI"]


def build_anomaly_image(sensor_name: str, index_bands: list) -> ee.Image:
    """Current-minus-pre period difference map (plan Sec.10.3/17, "recent pre/post anomaly
    maps"). Not the Mbokishi-normalized site_relative_anomaly from plan Sec.10.2 -- see the
    markdown note above."""
    current = period_records[(sensor_name, "current")]
    pre = period_records[(sensor_name, "pre")]
    diffs = [
        current.select(band).subtract(pre.select(band)).rename(f"{band}_diff") for band in index_bands
    ]
    return ee.Image.cat(diffs).clip(project_geom)


anomaly_maps = {
    sensor_name: build_anomaly_image(sensor_name, ANOMALY_INDEX_BANDS)
    for sensor_name in SENSOR_SPECS
    if (sensor_name, "pre") in period_records and (sensor_name, "current") in period_records
}
print(f"Built anomaly maps for: {sorted(anomaly_maps)}")

## 8. Zonal statistics -- build & export only

Same convention as Objective 1: everything below builds `ee.FeatureCollection`s and hands them
straight to `export_table_to_drive` -- no `.getInfo()`/pandas conversion happens in this
notebook. Download the exported CSVs from Drive and do the pandas joins/reference-normalized
anomaly calculation (plan Sec.10.2) locally, in a separate step (see Objective 1's own plotting
notebook for the established pattern).

In [ ]:
# mean/median/p10/p25/p75/p90/stdDev/count -- the plan's own recommended zonal-statistics
# schema (Sec.9.1/9.2); median and percentiles are used in reporting because they're less
# sensitive to outliers than the mean (plan Sec.9.2).
TS_REDUCER = (
    ee.Reducer.mean()
    .combine(ee.Reducer.median(), sharedInputs=True)
    .combine(ee.Reducer.percentile([10, 25, 75, 90]), sharedInputs=True)
    .combine(ee.Reducer.stdDev(), sharedInputs=True)
    .combine(ee.Reducer.count(), sharedInputs=True)
)

sensor_ts_stats_fc = {}
for sensor_name, spec in SENSOR_SPECS.items():
    sensor_records = {key: image for key, image in records.items() if key[0] == sensor_name}
    collection = ee.ImageCollection(list(sensor_records.values()))
    sensor_ts_stats_fc[sensor_name] = image_collection_to_region_stats_fc(
        collection=collection,
        regions_fc=sites_fc,
        bands=spec["index_bands"] + [VALID_OBS_BAND, "coverage_flag"],
        scale=spec["scale"],
        reducers=TS_REDUCER,
        image_properties=["sensor_group", "year", "season"],
        tile_scale=4,
    )

In [ ]:
sensor_period_stats_fc = {}
for sensor_name, spec in SENSOR_SPECS.items():
    sensor_periods = {key: image for key, image in period_records.items() if key[0] == sensor_name}
    collection = ee.ImageCollection(list(sensor_periods.values()))
    sensor_period_stats_fc[sensor_name] = image_collection_to_region_stats_fc(
        collection=collection,
        regions_fc=sites_fc,
        bands=spec["index_bands"] + [VALID_OBS_BAND, "coverage_flag"],
        scale=spec["scale"],
        reducers=TS_REDUCER,
        image_properties=["sensor_group", "period"],
        tile_scale=4,
    )

## 9. LandTrendr change-event workflow

Dry-season Landsat NBR, 1984-2025 -- the plan's primary segmentation input (Sec.11.2/11.3).
`eetools.landtrendr.run_landtrendr_from_aoi` builds the Roy-harmonized annual medoid collection
and runs LandTrendr in one call, and its default run parameters
(`eetools.constants.LANDTRENDR_DEFAULT_RUN_PARAMS`) already match the plan's own starting values
exactly -- no override needed (objective-1->2 handoff notes Sec.2). `get_change_map` already
handles the loss-positive orientation and both disturbance (`delta="loss"`) and recovery
(`delta="gain"`) extraction internally, resolving the plan's own Sec.11.5 gap (no
array-unpacking code needed here).

`eetools.landtrendr` now accepts any `INDEX_REGISTRY` index computable from the Landsat common
bands as a fit-to-vertices (FTV) band (generalized beyond the old NBR/NDVI/NDMI-only allowlist),
so `config.LANDTRENDR_FTV_INDICES` carries NDMI, MSAVI2, and BSI alongside the NBR segmentation
index, fulfilling the plan's own Sec.11.2 request in full.

In [ ]:
lt = run_landtrendr_from_aoi(
    project_geom,
    start_year=config.LANDTRENDR_YEAR_START,
    end_year=config.LANDTRENDR_YEAR_END,
    segmentation_index=config.LANDTRENDR_SEGMENTATION_INDEX,
    ftv_indices=config.LANDTRENDR_FTV_INDICES,
    start_day=config.LANDTRENDR_SEASON_DAYS[0],
    end_day=config.LANDTRENDR_SEASON_DAYS[1],
)

In [ ]:
def add_area_band(change_image: ee.Image) -> ee.Image:
    """Per-pixel area (ha), masked to the change image's own kept pixels -- lets the zonal
    stats below report total disturbed/recovered area per site via a sum reducer."""
    area_ha = ee.Image.pixelArea().divide(10000).updateMask(change_image.select("yod").mask())
    return change_image.addBands(area_ha.rename("area_ha"))


def mask_change_to_window(change_image: ee.Image, year_start: int, year_end: int) -> ee.Image:
    """Restrict a change map to pixels whose year-of-detection (yod) falls in [year_start,
    year_end] -- the plan's own "recent disturbance/recovery window" products (Sec.11.5)."""
    yod = change_image.select("yod")
    return change_image.updateMask(yod.gte(year_start).And(yod.lte(year_end)))


disturbance_change = add_area_band(get_change_map(lt, change_params={"delta": "loss"}))
recovery_change = add_area_band(get_change_map(lt, change_params={"delta": "gain"}))

recent_windows = config.LANDTRENDR_RECENT_WINDOWS
disturbance_recent = {
    f"{year_start}_{year_end}": mask_change_to_window(disturbance_change, year_start, year_end)
    for year_start, year_end in recent_windows["disturbance"]
}
recovery_recent = {
    f"{year_start}_{year_end}": mask_change_to_window(recovery_change, year_start, year_end)
    for year_start, year_end in recent_windows["recovery"]
}

In [ ]:
LT_STATS_BANDS = ["yod", "mag", "dur", "rate", "dsnr", "area_ha"]

lt_change_images = {
    "disturbance_full": disturbance_change,
    "recovery_full": recovery_change,
    **{f"disturbance_recent_{label}": image for label, image in disturbance_recent.items()},
    **{f"recovery_recent_{label}": image for label, image in recovery_recent.items()},
}
lt_stats_collection = ee.ImageCollection(
    [image.set({"change_type": name}) for name, image in lt_change_images.items()]
)
lt_event_summary_fc = image_collection_to_region_stats_fc(
    collection=lt_stats_collection,
    regions_fc=sites_fc,
    bands=LT_STATS_BANDS,
    scale=LANDSAT_SCALE,
    reducers=(
        ee.Reducer.mean()
        .combine(ee.Reducer.sum(), sharedInputs=True)
        .combine(ee.Reducer.count(), sharedInputs=True)
    ),
    image_properties=["change_type"],
    tile_scale=4,
)

### Export zonal-statistics and LandTrendr event-summary tables to Drive

In [ ]:
STATS_TABLES = {
    f"{sensor_name}_site_timeseries_annual_seasonal": stats_fc
    for sensor_name, stats_fc in sensor_ts_stats_fc.items()
} | {
    f"{sensor_name}_site_summary_by_period": stats_fc
    for sensor_name, stats_fc in sensor_period_stats_fc.items()
} | {
    "landtrendr_event_summary_by_site": lt_event_summary_fc,
}

for table_name, table_fc in STATS_TABLES.items():
    export_table_to_drive(
        collection=table_fc,
        description=table_name,
        folder=EXPORT_FOLDER,
        fileNamePrefix=f"{table_name}",
    )

print(f"Started {len(STATS_TABLES)} table export tasks to Drive folder '{EXPORT_FOLDER}'.")

### Optional -- CHIRPS annual rainfall QA table

Single AOI-wide table, not per-site -- CHIRPS' ~5.5 km native resolution is coarser than the
separation between these four sites, so a per-site breakdown wouldn't add real spatial
information. Supports the plan's rainfall-confounding check (Sec.16.D): if every site declines
in the same year, cross-check against a rainfall dip here before reading it as degradation.

**Manual step, optional** -- run to export the rainfall QA table.

In [ ]:
export_total_rainfall_table(
    start_date=f"{config.LANDSAT_YEAR_START}-01-01",
    end_date=f"{YEAR_END + 1}-01-01",
    aoi=project_geom,
    temporal_scale="annual",
    export_folder=EXPORT_FOLDER,
    file_prefix="chirps_annual_rainfall_1984_2025",
)
print("Started the CHIRPS annual rainfall export task.")

## 10. Visual QA

In [ ]:
Map = geemap.Map()
Map.centerObject(project_geom, 12)
Map.addLayer(sites_fc.style(**SITES_VIS_PARAMS), {}, "Study sites")
Map.addLayer(
    period_records[("landsat", "current")].select("MSAVI2"),
    {"min": 0, "max": 0.8, "palette": ["a50026", "fee08b", "1a9850"]},
    "Landsat MSAVI2 (current 2022-2025)",
)
Map.addLayer(
    anomaly_maps["landsat"].select("MSAVI2_diff"),
    {"min": -0.2, "max": 0.2, "palette": ["d73027", "ffffbf", "1a9850"]},
    "Landsat MSAVI2 anomaly (current - pre)",
    False,
)
Map.addLayer(
    disturbance_change.select("yod"),
    {"min": 1985, "max": 2025, "palette": ["ffffb2", "fd8d3c", "bd0026"]},
    "LandTrendr disturbance year (dry-season NBR)",
    False,
)
Map.addLayer(
    recovery_change.select("yod"),
    {"min": 1985, "max": 2025, "palette": ["edf8b1", "7fcdbb", "2c7fb8"]},
    "LandTrendr recovery year (dry-season NBR)",
    False,
)
Map

## 11. Export rasters to Google Drive

Split into groups so each batch can be reviewed in the EE Task Manager before the next, per
Objective 1's export-batching convention. Groups 11a-11d cover the plan's own "minimum export
set" (Sec.12) and run by default; Group 11e (full per-year seasonal/annual composites for every
sensor) is optional and deferred, mirroring Objective 1's own per-year annual group -- it is far
larger in volume than anything else here (Landsat alone is 42 years x 3 seasons) and isn't part
of the plan's minimum set.

### 11a. Trend maps

In [ ]:
EXPORT_TASKS_TRENDS = [
    (image, f"trend_maps/{sensor}_{index_band}_slope_{label}_project", SENSOR_SPECS[sensor]["scale"])
    for (sensor, index_band, label), image in trend_maps.items()
]
print(f"{len(EXPORT_TASKS_TRENDS)} trend-map export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_TRENDS, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} trend-map export tasks.")

### 11b. Pre/post anomaly (difference) maps

In [ ]:
EXPORT_TASKS_ANOMALY = [
    (
        image,
        f"anomaly_maps/{sensor_name}_pre_2016_2021_vs_current_2022_2025_project",
        SENSOR_SPECS[sensor_name]["scale"],
    )
    for sensor_name, image in anomaly_maps.items()
]
print(f"{len(EXPORT_TASKS_ANOMALY)} anomaly-map export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_ANOMALY, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} anomaly-map export tasks.")

### 11c. LandTrendr disturbance/recovery maps

In [ ]:
LT_EXPORT_BANDS = ["yod", "mag", "dur", "rate", "dsnr"]

EXPORT_TASKS_LANDTRENDR = [
    (image.select(LT_EXPORT_BANDS), f"landtrendr/LT_NBR_dry_1984_2025_{name}_project", LANDSAT_SCALE)
    for name, image in lt_change_images.items()
]
print(f"{len(EXPORT_TASKS_LANDTRENDR)} LandTrendr export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_LANDTRENDR, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} LandTrendr export tasks.")

### 11d. Valid-observation-count QA rasters (period composites)

Period-level rather than every per-year composite, to keep the default export volume bounded --
per-year QA rasters are available in Group 11e below if a later analysis step needs them.

In [ ]:
EXPORT_TASKS_QA = [
    (
        image.select(VALID_OBS_BAND),
        (
            f"qa/{sensor_name}_valid_obs_{period_name}_"
            f"{config.PERIODS[period_name][0]}_{config.PERIODS[period_name][1]}_project"
        ),
        SENSOR_SPECS[sensor_name]["scale"],
    )
    for (sensor_name, period_name), image in period_records.items()
]
print(f"{len(EXPORT_TASKS_QA)} valid-observation QA export tasks staged.")

**Manual step** -- review the count above, then run to start this batch.

In [ ]:
started = export_image_list_to_drive(EXPORT_TASKS_QA, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS)
print(f"Started {len(started)} valid-observation QA export tasks.")

### 11e. Full per-year seasonal & annual composites -- optional, deferred

Not part of the plan's minimum export set (Sec.12); every sensor's full per-year, per-season
index composite as one multiband GeoTIFF each. Skipped by default to reduce EECU/server load and
Drive export volume -- export only if a later analysis step needs individual-year rasters rather
than the zonal-statistics tables already exported above.

In [ ]:
EXPORT_TASKS_FULL_COMPOSITES = [
    (
        image.select(SENSOR_SPECS[sensor_name]["index_bands"] + [VALID_OBS_BAND]),
        f"{sensor_name}_composites/{sensor_name}_indices_{season}_{year}_project",
        SENSOR_SPECS[sensor_name]["scale"],
    )
    for (sensor_name, year, season), image in records.items()
]
print(f"{len(EXPORT_TASKS_FULL_COMPOSITES)} per-year composite export tasks staged (optional).")

**Manual step, optional** -- only run this if a later analysis step needs individual-year
rasters; review the count above first (this is the largest export group by far).

In [ ]:
started = export_image_list_to_drive(
    EXPORT_TASKS_FULL_COMPOSITES, aoi=project_geom, folder=EXPORT_FOLDER, crs=CRS
)
print(f"Started {len(started)} per-year composite export tasks.")

## 12. Limitations & next steps

- Landsat provides the longest time series (1984-present) but at coarser 30 m resolution; HLS
  (2015-present) adds temporal density at the same 30 m; Sentinel-2 (2017-present) adds 10 m
  detail and red-edge indices but only for the recent record. Per the plan (Sec.16.C), these are
  interpreted as three separate sensor-group time series, not merged into one continuous line.
- Classification-free by design: this notebook reports continuous index values, not a
  categorical class. Cross-check index trends and LandTrendr events against Objective 1's
  Dynamic World transitions before reporting a conversion/degradation claim (plan Sec.15).
- `valid_obs_count`/`coverage_flag` thresholds (`config.MIN_OBS`) are starting values, same
  caveat as Objective 1's `DW_MIN_OBS_*` -- HLS in particular borrows the Landsat thresholds
  since the plan itself doesn't specify HLS-specific ones (Sec.7).
- LandTrendr disturbance/recovery magnitude thresholds are not yet set (plan Sec.11.5 recommends
  starting with the upper 20% of magnitude values, tuned after visual inspection) -- the
  exported `mag`/`rate`/`dsnr` bands are provided unfiltered beyond the MMU cleanup already
  applied by `get_change_map`'s default `mmu` parameter; apply a magnitude threshold once
  high-resolution imagery comparison (plan Sec.16.E) has calibrated one.
- Landsat 7 scenes after ~2017 have known orbit-drift-driven solar-geometry changes (on top of
  the older SLC-off gaps) -- cross-check any disturbance/recovery event dated 2017-2022 against
  L8/L9 coverage for the same pixel before attributing it to real vegetation change (plan
  Sec.16.B vault note).
- Reference-normalized (site-minus-Mbokishi) anomalies and rolling-window trend charts (plan
  Sec.10.2/14) are pandas post-processing on the exported CSVs, done in a separate step -- not
  attempted in this notebook (same division of labor as Objective 1's own plotting notebook).
- Cross-check recovery/degradation signals here against Objective 1's Dynamic World categorical
  transitions, and validate high-change hotspots visually against high-resolution imagery,
  before finalizing any reporting claim (plan Sec.15/16.E).